# Stage 1: Chốt schema + refactor skeleton

## 1. Load Bronze Dataset

In [ ]:
from pathlib import Path

import fiftyone as fo

from z_photos.conf import Config
from z_photos.datasets import BronzeField, build_bronze_dataset

cfg = Config(
    data_root=Path("../data"),
    seed=42,
)

bronze_dataset = build_bronze_dataset("z_photos-bronze", bronze_dir=cfg.bronze_dir)
print(f"Bronze dataset size: {len(bronze_dataset)}")

Bronze dataset size: 321


## 2. Refactor: Update Bronze schema with flags

In [ ]:
# Add missing flags to Bronze layer to align with Stage 1 contract
def init_boolean_flag(dataset, field_name: str, default: bool = False):
    if not dataset.has_sample_field(field_name):
        dataset.add_sample_field(field_name, fo.BooleanField)
        dataset.set_values(field_name, [default] * len(dataset))


init_boolean_flag(bronze_dataset, BronzeField.IS_EXACT_DUP)
init_boolean_flag(bronze_dataset, BronzeField.IS_NEAR_DUP)
init_boolean_flag(bronze_dataset, BronzeField.IS_LEAKY)

# Currently, `main.ipynb` marked leaky samples using tags
# Let's port that knowledge over to the newly established boolean field if available
for sample in bronze_dataset:
    changed = False
    if "leaky" in sample.tags:
        sample[BronzeField.IS_LEAKY] = True
        sample.tags.remove("leaky")
        changed = True

    if changed:
        sample.save()

print("Bronze schema updated.")

Bronze schema updated.


## 3. Clone for Silver and Gold Views

In [ ]:
from fiftyone import ViewField


def get_or_create_clone(dataset, view, clone_name: str):
    if clone_name in fo.list_datasets():
        # Clean up existing ones for this experiment if we want to rebuild
        fo.delete_dataset(clone_name)
    return view.clone(clone_name, persistent=True)


# 1. Silver Train (from train tag)
silver_train_view = bronze_dataset.match_tags("train")
silver_train = get_or_create_clone(
    bronze_dataset, silver_train_view, "z_photos-silver-train"
)
print(f"Silver Train created: {len(silver_train)} samples")

# 2. Gold Eval Clean (from test tag, excluding leaky)
gold_eval_clean_view = bronze_dataset.match_tags("test").match(
    ~ViewField(BronzeField.IS_LEAKY)
)
gold_eval_clean = get_or_create_clone(
    bronze_dataset, gold_eval_clean_view, "z_photos-gold-eval-clean"
)
print(f"Gold Eval Clean created: {len(gold_eval_clean)} samples")

Silver Train created: 261 samples
Gold Eval Clean created: 50 samples


## 4. Audit Detectors: Exact Duplicates

In [ ]:
# --- Audit Detectors: Exact Duplicates ---
import fiftyone.brain as fob

# 1. Compute Exact Duplicates on Bronze
exact_dups = fob.compute_exact_duplicates(bronze_dataset)

# Mark them in Bronze
dup_ids = [did for dups in exact_dups.values() for did in dups]
bronze_dataset.select(dup_ids).set_values(
    BronzeField.IS_EXACT_DUP, [True] * len(dup_ids)
)

# 2. Re-propagate to Silver Train (silver_train là bản clone trước khi có field này)
# Ta dùng merge_samples để đồng bộ các field audit từ Bronze sang Silver
silver_train.merge_samples(
    bronze_dataset, fields=[BronzeField.IS_EXACT_DUP, BronzeField.IS_LEAKY]
)

print(f"Audit fields updated. Exact dups found: {len(dup_ids)}")

Computing filehashes...
 100% |█████████████████| 321/321 [162.3ms elapsed, 0s remaining, 2.0K samples/s]     
Audit fields updated. Exact dups found: 14


# Stage 2: SigLIP2 Cleaning for Silver Train

In [ ]:
from z_photos.clip_cleaner import SigLIP2Cleaner, apply_clip_cleaner

# Initialize cleaner (using auto device for speed)
cleaner = SigLIP2Cleaner(device_map="auto")

# Apply cleaner to silver_train
# This populates: siglip2_emb, zs_score, lr_oof_score, clean_score, sample_weight_prelim
apply_clip_cleaner(silver_train, cleaner=cleaner, alpha=0.4)

# Print some results
print("\nSample clean scores:")
for sample in silver_train.limit(5):
    print(f"File: {Path(sample.filepath).name} | Clean Score: {sample.clean_score:.4f}")

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

Extracting embeddings: 100%|██████████| 21/21 [00:47<00:00,  2.25s/it]


Computing Zero-shot scores...
Computing LR OOF scores...
Combining scores...
Cleaning scores applied to 321 samples.

Sample clean scores:
File: 0000_07bd76f8d5ed.jpg | Clean Score: 0.9998
File: 0001_dc637aea6f5d.jpg | Clean Score: 0.9994
File: 0002_5083c5b917aa.jpg | Clean Score: 0.9979
File: 0003_4b315d3e7d10.jpg | Clean Score: 0.9995
File: 0004_001d05b6ab97.jpg | Clean Score: 0.8773


# Stage 3: Materialize Gold Train

In [ ]:
# --- Stage 3: Materialize Gold Train ---
from fiftyone import ViewField as F

from z_photos.datasets import BronzeField, GoldField, SilverField

# Policy:
# 1. Drop cứng: IS_EXACT_DUP hoặc IS_LEAKY
# 2. Giữ nhưng down-weight (sample_weight): clean_score

# 1. Clone từ Silver Train sang Gold Train
gold_train = get_or_create_clone(silver_train, silver_train, "z_photos-gold-train")

# 2. Cấu hình gold fields
if not gold_train.has_sample_field(GoldField.KEEP_FOR_TRAIN):
    gold_train.add_sample_field(GoldField.KEEP_FOR_TRAIN, fo.BooleanField)

if not gold_train.has_sample_field(GoldField.SAMPLE_WEIGHT):
    gold_train.add_sample_field(GoldField.SAMPLE_WEIGHT, fo.FloatField)

# 3. Áp dụng logic Policy
# Mặc định keep_for_train=True và sample_weight=clean_score
gold_train.set_values(GoldField.KEEP_FOR_TRAIN, [True] * len(gold_train))
clean_scores = gold_train.values(SilverField.CLEAN_SCORE)
gold_train.set_values(GoldField.SAMPLE_WEIGHT, clean_scores)

# Filter: drop samples based on flags
bad_samples = gold_train.match(
    (F(BronzeField.IS_EXACT_DUP)) | (F(BronzeField.IS_LEAKY))
)
bad_samples.set_values(GoldField.KEEP_FOR_TRAIN, [False] * len(bad_samples))
# Also set weight=0 for bad samples to be safe
bad_samples.set_values(GoldField.SAMPLE_WEIGHT, [0.0] * len(bad_samples))

# Stats
keep_view = gold_train.match(F(GoldField.KEEP_FOR_TRAIN))
print("Gold Train Stats:")
print(f"- Total: {len(gold_train)}")
print(f"- Dropped (Exact Dup/Leaky): {len(bad_samples)}")
print(f"- Final keep samples for training: {len(keep_view)}")

weights = keep_view.values(GoldField.SAMPLE_WEIGHT)
if weights:
    print(f"- Min clean weight: {min(weights):.4f}")
    print(f"- Max clean weight: {max(weights):.4f}")
    print(f"- Avg clean weight: {sum(weights) / len(keep_view):.4f}")

gold_train.save()
print("\nStage 3 complete. Gold Train dataset ready for Stage 4 (Trainer).")

Gold Train Stats:
- Total: 321
- Dropped (Exact Dup/Leaky): 31
- Final keep samples for training: 290
- Min clean weight: 0.0000
- Max clean weight: 0.9998
- Avg clean weight: 0.6728

Stage 3 complete. Gold Train dataset ready for Stage 4 (Trainer).
